# Simulación de portafolio con Cholesky
María Fernanda Gómez Flores

AAPL, MSFT y NVDA. Simulación Monte Carlo de 10,000 escenarios para un horizonte de 1 año.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

### Datos históricos y rendimientos
Descargamos los precios de cierre de las tres acciones y calculamos sus rendimientos logarítmicos diarios.

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA"]
prices = yf.download(tickers=tickers, start="2023-11-01", end="2026-11-01")["Close"]
prices = prices[tickers].dropna()
rets = np.log(prices / prices.shift(1)).dropna()
rets

### Matriz de correlación
Identificamos la relación entre los rendimientos de las acciones.

In [ ]:
rets.corr()

### Matriz de covarianzas
Calculamos la covarianza diaria de los rendimientos, que utilizaremos para generar escenarios correlacionados.

In [ ]:
Sigma = rets.cov()
Sigma

### Descomposición de Cholesky
Descomponemos la matriz de covarianzas para transformar números normales independientes en variables correlacionadas.

In [ ]:
L = np.linalg.cholesky(Sigma)
L

### Rendimientos promedio
Calculamos los rendimientos logarítmicos promedio y los anualizamos considerando 252 días bursátiles.

In [ ]:
mu = rets.mean().values * 252
mu

### Simulación de números aleatorios
Generamos 10,000 escenarios de tres variables normales independientes.

In [ ]:
M = 10000
rng = np.random.default_rng(42)
epsilon = rng.standard_normal((len(tickers), M))
epsilon

### Rendimientos aleatorios correlacionados
Aplicamos Cholesky para que los movimientos simulados conserven la estructura de covarianzas observada.

In [ ]:
Z = L @ epsilon
Z.T

### Simulación de precios dentro de un año
Calculamos el precio futuro de cada acción con movimiento browniano geométrico. Como Cholesky ya incorpora la volatilidad diaria, no volvemos a multiplicar el componente aleatorio por ella.

In [ ]:
T = 1
n_dias = 252
S0 = prices.iloc[-1].values
mu = rets.mean().values * n_dias
vol = rets.std().values * np.sqrt(n_dias)

ST = S0 * np.exp(
    (mu - 0.5 * vol**2) * T
    + np.sqrt(n_dias * T) * Z.T
)
ST = pd.DataFrame(ST, columns=tickers)
ST.head()

### Valor futuro del portafolio
Suponemos un capital inicial de $100,000 distribuido en partes iguales entre las tres acciones. Calculamos cuántas acciones compramos inicialmente y el valor final del portafolio en cada simulación.

In [ ]:
V0 = 100000
w = np.array([1/3, 1/3, 1/3])
acciones = V0 * w / S0
VT = ST.to_numpy() @ acciones

print(f"Capital inicial: ${V0:,.2f}")
print(f"Valor futuro promedio: ${VT.mean():,.2f}")

### Histograma del valor futuro
Primero graficamos los valores simulados del portafolio **antes de restar el capital inicial**. La línea roja señala los $100,000 invertidos.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(VT, bins=50, edgecolor="black")
plt.axvline(V0, color="red", linestyle="--", label="Capital inicial")
plt.title("Distribución del valor futuro del portafolio")
plt.xlabel("Valor futuro ($)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

### Ganancias y pérdidas
Restamos el capital inicial al valor final de cada simulación.

$$P\&L = V_T - V_0$$

Los resultados negativos representan pérdidas y los positivos ganancias.

In [ ]:
PnL = VT - V0
print(f"Ganancia o pérdida promedio: ${PnL.mean():,.2f}")
print(f"Probabilidad simulada de pérdida: {(PnL < 0).mean():.2%}")
PnL

### Histograma de ganancias y pérdidas
Graficamos nuevamente la distribución, ahora **después de restar el capital inicial**. El cero representa recuperar exactamente la inversión.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(PnL, bins=50, edgecolor="black")
plt.axvline(0, color="red", linestyle="--", label="Ganancia/Pérdida = 0")
plt.title("Distribución de ganancias y pérdidas del portafolio")
plt.xlabel("Ganancia / Pérdida ($)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()